Bird Species Image Classifier: Data Preparation and Exploration

Welcome to the first phase of building our deep learning image classifier! Before we ever define a neural network or write a training loop, we must understand and prepare our environment and our data.

In this notebook, we will progress through five essential steps:

    System Overview: Checking our hardware constraints.

    Basic Data Analysis: Understanding our dataset's distribution and visualizing our inputs.

    Preparing the Dataset: Building a custom PyTorch Dataset class.

    Train/Test Splitting: Dividing our data to ensure our model can be evaluated fairly.

    Building DataLoaders: Creating the pipeline that feeds batches of images to our model.

Let's get started!

Chapter 0: System Overview

Before building a deep learning model, it is crucial to know the hardware you are working with. Image datasets can be massive, and neural networks require significant memory. By checking our RAM, storage, and CPU cores, we can plan our batch_size and avoid crashing our computer with "Out of Memory" errors.

We can run standard terminal commands directly inside a Jupyter Notebook by prefixing them with an exclamation mark (!).

In [ ]:
# Check RAM usage
print("--- RAM USAGE ---")
!free -h

# Check Storage
print("\n--- STORAGE ---")
!df -h

# Look for CPU cores
print("\n--- CPU INFO ---")
!lscpu | grep "CPU(s):"

# Note: If you are running this on Windows, the above Linux commands might not work. 
# If they fail, don't worry! You can just skip this cell and move to the data.

Chapter 1: Basic Data Analysis

The most important component of any deep learning model is the data. If we feed garbage into our model, we will get garbage out.

Our first step is to check the label distribution. We need to know how many images we have for each bird species.

    Balanced Dataset: The percentage of samples for each label is roughly equal.

    Imbalanced Dataset: Some labels have significantly more samples than others. This would require special modeling techniques (like weighted loss functions) to prevent the model from just guessing the most common bird every time.

After checking the distribution, we will visualize the images. Seeing the data helps us understand the variance in lighting, angles, and backgrounds our model will have to learn from.

In [ ]:
import glob, os
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import random

# Point to the local folder where your images are stored
data_path = "./birds_data/*"
data = defaultdict(list)

# Iterate through directories and grab all image files
for dir_ in glob.glob(data_path):
    # Grab jpg, jpeg, and png files
    images = glob.glob(f"{dir_}/*.jpg") + glob.glob(f"{dir_}/*.jpeg") + glob.glob(f"{dir_}/*.png")
    for file in images:
        data[Path(dir_).name].append(Path(file))

# Calculate class counts and percentages to check if data is balanced
class_counts = list(map(lambda x: len(x), data.values()))
total_samples = sum(class_counts)
class_percentage = [i / total_samples for i in class_counts]

print("Class Distribution:")
for name, count, percent in zip(data.keys(), class_counts, class_percentage):
    print(f"Species: {name} | Count: {count} | Percentage: {percent*100:.2f}%")

print("\nObservation: The dataset is relatively balanced.")

In [ ]:
def plot_image(images, title):
    """Plots 10 random images from a given list of image paths."""
    # Choose 10 random images from the list
    random_images = random.sample(images, 10)

    # Set up a Matplotlib figure with subplots (2 rows, 5 columns)
    fig, axes = plt.subplots(2, 5, figsize=(12, 6))
    fig.suptitle(title, fontsize=16)

    # Iterate over the subplots and display the images
    for i, ax in enumerate(axes.flatten()):
        # Read the image using Matplotlib's imread function
        img = mpimg.imread(random_images[i])
        ax.imshow(img)
        ax.axis('off') # Hide the axes ticks for a cleaner look

    plt.tight_layout()
    plt.show()

# Visualize 10 random images from each class
for key, value in data.items():
    plot_image(value, key)

Chapter 2: Prepare the Dataset

Now that we understand our data, we need to translate it into a format that PyTorch can understand.

A PyTorch Dataset is a container that organizes your data. To create a custom dataset, we inherit from PyTorch's Dataset class and implement three mandatory methods:

    __init__: Runs once when we instantiate the Dataset. We use it to store our image paths and labels.

    __len__: Returns the total number of samples in our dataset.

    __getitem__: The most important method. It tells PyTorch how to fetch a single piece of data (one image and its label) at a specific index.

Image Transformations:
Neural networks require fixed-size inputs and work best with normalized numbers. Inside our dataset, we apply a series of transformations:

    Resize((224, 224)): Squashes or stretches all images to the same square size.

    ToTensor(): Converts the image from a Pillow format (pixels 0-255) into a PyTorch Tensor (floating point numbers 0.0-1.0).

    Normalize(): Shifts the pixel values to have a mean of 0 and a standard deviation of 1, which helps the neural network converge (learn) faster.

In [ ]:
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image

class BirdDataset(Dataset):
    def __init__(self, data_dict, transform=None):
        self.data_dict = data_dict
        self.transform = transform

        # Flatten the dictionary into a list of (class_name, image_path) tuples
        self.samples = [(class_name, image_name) for class_name, image_names in data_dict.items() for image_name in image_names]
        
        # Create mapping dictionaries to convert string labels (e.g., 'Eagle') to numeric indices (e.g., 0)
        self.classes_to_index = {class_name: index for index, class_name in enumerate(self.data_dict.keys())}
        self.index_to_classes = {index: class_name for class_name, index in self.classes_to_index.items()}
        
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        # 1. Fetch the class name and image path based on the index
        class_name, img_path = self.samples[idx]

        # 2. Load image using Pillow and ensure it has 3 color channels (RGB)
        img = Image.open(img_path).convert("RGB")

        # 3. Apply transformations (resize, to tensor, normalize)
        if self.transform:
            img = self.transform(img)

        # 4. Return the processed image tensor and the numeric label tensor
        return img, torch.tensor(self.classes_to_index[class_name])


# Define the sequence of transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),  
    transforms.ToTensor(), 
    # Standard ImageNet normalization values
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
])

# Instantiate our custom dataset
bird_dataset = BirdDataset(data, transform)

print(f"Total samples in dataset: {len(bird_dataset)}")

Chapter 3: Split into Train and Test Dataset

If we train our model on all our images, we have no way to know if it actually learned the defining features of a bird, or if it just memorized the specific images we showed it.

To solve this, we split our data:

    Training Set (80%): The data the model studies to learn patterns.

    Testing Set (20%): Unseen data used to evaluate the model's true accuracy.

Crucial Rule: Test data must never leak into the training set, otherwise our evaluation metrics will be fake.

In [ ]:
from torch.utils.data import random_split

# Calculate lengths for an 80/20 split
test_no = int(0.2 * len(bird_dataset))
train_no = len(bird_dataset) - test_no

# Randomly split the dataset
trainSet, testSet = random_split(bird_dataset, (train_no, test_no))

print(f"Length of Training Set: {len(trainSet)}")
print(f"Length of Testing Set: {len(testSet)}")

Chapter 4: Build DataLoaders

Our Dataset is great at fetching one image at a time. However, neural networks learn best (and train much faster on hardware) when they are fed groups of images at the same time. These groups are called batches.

The PyTorch DataLoader takes our Dataset and wraps it in an iterator that handles:

    Batching: Grouping images together (e.g., 32 images per batch).

    Shuffling: Mixing up the order of the training images every epoch so the model doesn't memorize the sequence.

Notice below that when we print the shape of a single batch of images, the format is [Batch_Size, Color_Channels, Height, Width].

In [ ]:
from torch.utils.data import DataLoader

# Define how many images the model sees at once
batch_size = 32

# Create the DataLoaders
# We shuffle the train loader to ensure randomness, but we don't need to shuffle the test loader
trainLoader = DataLoader(trainSet, batch_size=batch_size, shuffle=True) 
testLoader = DataLoader(testSet, batch_size=batch_size, shuffle=False)

print(f"Number of batches in trainLoader: {len(trainLoader)}")
print(f"Number of batches in testLoader: {len(testLoader)}")

# Verify the shape of a single batch
images, labels = next(iter(trainLoader))
print(f"Batch Image Tensor Shape: {images.shape}") # Expected: [32, 3, 224, 224]
print(f"Batch Label Tensor Shape: {labels.shape}") # Expected: [32]

# Group them into a dictionary for easy access later
data_loader = {"train": trainLoader, "test": testLoader}

Building the Image Classifier Model

Now that our data is prepared and flowing through our DataLoaders, we need a brain to process it.

Instead of building a Convolutional Neural Network (CNN) from scratch, we will use a powerful technique called Transfer Learning.

Think of transfer learning like this: instead of teaching a toddler how to fly a commercial jet, you take a professional pilot who already knows how to fly a Boeing 747, and you just teach them the specific controls of an Airbus A320.

In computer vision, we do this by downloading a model that was pre-trained on a massive dataset called ImageNet (millions of images, 1000 categories). The model has already spent weeks on supercomputers learning how to detect edges, shapes, textures, and complex patterns. We will take this pre-trained model and just swap out the very last layer so it outputs our specific bird classes instead of the original 1000 ImageNet classes.

Let's build it!

Chapter 5: Import Pretrained ResNet Architecture

The specific model we are using is called ResNet-18 (Residual Network, 18 layers deep).

Before ResNet, making neural networks deeper often made them worse because the learning signal (the gradient) would die out before reaching the earlier layers—a problem known as the "vanishing gradient." ResNet solved this by introducing "skip connections," which act like shortcut highways that allow the signal to bypass certain layers, making it possible to train incredibly deep and accurate networks.

Let's download the pre-trained ResNet-18 from PyTorch's library.

In [ ]:
import torch
from torchvision.models import resnet18, ResNet18_Weights

# Download the ResNet-18 model, including the weights (the "knowledge") it learned from ImageNet
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

# Calculate the total number of parameters (the "synapses" in our brain)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total Number of Parameters in ResNet-18: {total_params:,}")

# Let's see what the architecture looks like
# (It will print a long list of Convolutional layers, Batch Normalizations, and ReLUs)
print("\nModel Architecture loaded successfully!")

Chapter 6: Freeze the Layer

Right now, every parameter in our downloaded model is set to update and learn when we train it. However, the early and middle layers of this model already act as perfect "feature extractors"—they already know how to spot eyes, feathers, and textures. We don't want to accidentally ruin that knowledge!

So, we are going to freeze the "backbone" of the model.

Freezing means we tell PyTorch: "Do not calculate gradients or update the weights for these layers." This saves massive amounts of memory and time during training, and protects the pre-trained knowledge.

After freezing the backbone, we will temporarily replace the very last layer (the Fully Connected or fc layer) with a blank Identity layer, which does absolutely nothing. It just passes the data straight through. This prepares the model for our custom head in the next step.

In [ ]:
# 1. Freeze the entire model backbone
for param in model.parameters():
    param.requires_grad = False # Turn off learning for these layers

# Verify freezing worked (Should print False, meaning no gradients are active)
print(f"Are any original parameters still learning? : {any(p.requires_grad for p in model.parameters())}")

# 2. Replace the final classifying layer with a blank Identity layer
# We do this to strip away the original 1000-class ImageNet output
model.fc = torch.nn.Identity()

# Let's test the frozen backbone with a dummy batch of images
# We pass a random tensor shaped [Batch=32, Channels=3, Height=224, Width=224]
with torch.no_grad():
    dummy_input = torch.randn(32, 3, 224, 224)
    features_output = model(dummy_input)
    
print(f"Shape of extracted features before our custom head: {features_output.shape}") 
# Expected: [32, 512] (32 images, each reduced to 512 core features)

Chapter 7: Modify the Architecture

Now we have a frozen backbone that outputs 512 highly optimized features for every image.

Our job is to build a new "head" (a sequence of Fully Connected layers) that takes those 512 features and narrows them down to our specific number of bird classes. Because we are adding these layers from scratch, PyTorch will naturally leave requires_grad=True for them. When we finally start training, only this new head will learn!

We will use a sequence of:

    Linear Layer: To compress the 512 features.

    ReLU: The activation function to introduce non-linearity.

    BatchNorm1d: To stabilize the learning process by normalizing the batch.

    Dropout: A regularization technique that randomly turns off some neurons (e.g., 25% of them) during training to prevent the model from memorizing the data (overfitting).

    Final Linear Layer: To output our exact number of bird classes

In [ ]:
def create_head(num_in_features, num_classes, dropout_prob=0.25):
    """Creates a custom neural network head for classification."""
    
    # We will step down from 512 -> 128 -> Number of Bird Classes
    hidden_features = num_in_features // 4 
    
    layers = [
        torch.nn.Linear(num_in_features, hidden_features),
        torch.nn.ReLU(),
        torch.nn.BatchNorm1d(hidden_features),
        torch.nn.Dropout(dropout_prob),
        torch.nn.Linear(hidden_features, num_classes)
    ]
    
    # Bundle the layers into a single Sequential block
    return torch.nn.Sequential(*layers)

# Get the number of output features from our frozen backbone (which we found was 512)
num_features = features_output.size(1) 

# Get the number of classes from our dataset (Should be 10 for this project)
num_classes = len(bird_dataset.classes_to_index) 

# Create the new head and attach it to our model
top_head = create_head(num_features, num_classes) 
model.fc = top_head 

print("New classification head successfully attached to the model!")

Chapter 8: Test the Correctness

Before we write the training loop, we must test the plumbing. We need to ensure that our real data can successfully flow out of our DataLoader, into our model, and come out the other side as a prediction.

A few PyTorch rules we are applying here:

    Devices: Tensors and Models must live on the same hardware (either both on the CPU, or both on the GPU). We use .to(device) to handle this.

    torch.no_grad(): Since we are just testing the forward pass and not actually training yet, we use this context manager to tell PyTorch not to waste memory tracking gradients.

Let's push one real batch of bird images through the model.

In [ ]:
# Define our device (Use GPU if available, otherwise fallback to CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Move the entire model to the chosen device
model = model.to(device)

# Turn on evaluation mode (this changes how layers like Dropout and BatchNorm behave)
model.eval()

# Let's test one real batch from our training loader
with torch.no_grad():
    # Grab the next batch of images and their true labels
    input_data, labels = next(iter(trainLoader))
    
    # Move the data to the same device as the model
    input_data = input_data.to(device)
    labels = labels.to(device)
    
    # Pass the images through the model to get raw predictions (logits)
    predictions = model(input_data)
    
    print(f"\nShape of input batch: {input_data.shape}")
    print(f"Shape of model predictions: {predictions.shape}") 
    # Expected: [32, 10] (32 images, 10 class probabilities each)
    print(f"Shape of true labels: {labels.shape}")

print("\nSuccess! The data flows perfectly from the DataLoader through the modified ResNet model.")

Training the Model

We have our data ready, and our ResNet-18 model is built. Now, we need to create the engine that drives the learning process.

Training a neural network requires three main components:

    The Model: Makes a prediction.

    The Loss Function (The Grader): Compares the model's prediction to the true label and calculates how "wrong" the model is.

    The Optimizer (The Adjuster): Takes that "wrongness" score and adjusts the model's internal weights to make a better prediction next time.

Chapter 9: Define the Optimizer and Loss Function

For our "Grader," we are using CrossEntropyLoss. This is the standard loss function for multi-class classification problems (where an image can only belong to one bird species). Under the hood, it applies a Softmax function to turn the model's raw outputs into probabilities (e.g., "I am 90% sure this is an eagle, and 10% sure it's a robin").

For our "Adjuster," we are using the Adam Optimizer. Adam stands for Adaptive Moment Estimation. It is an incredibly popular optimizer because it automatically adapts the "learning rate" (how big of a step it takes when adjusting weights) for each individual parameter, making training much faster and more stable than traditional Stochastic Gradient Descent (SGD).

In [ ]:
import torch.optim as optim
import torch.nn as nn

# Define the Loss Function (The Grader)
criterion = nn.CrossEntropyLoss()

# Define the Optimizer (The Adjuster)
# We only pass in the model parameters that require gradients (our new custom head)
# lr=0.001 is the starting learning rate
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Loss function and Optimizer successfully initialized!")

Chapter 10 & 11: The Training and Inference Loop

The training loop is the heartbeat of a machine learning project. It is a loop that runs for a set number of epochs (one epoch = the model has seen the entire dataset exactly once).

We will combine the training loop (learning) and the inference loop (testing) into one powerful function.

Here is the anatomy of what happens inside this loop for each batch of images:

    Mode Switch: We switch the model to .train() mode for learning, and .eval() mode for testing. This changes how layers like Dropout behave.

    Forward Pass: The model looks at the images and makes a guess (output = model(data)).

    Calculate Loss: The criterion grades the guess (loss = criterion(output, target)).

    Backward Pass (loss.backward()): Train mode only. The math magic happens here. PyTorch calculates the gradients (which direction to turn the knobs to lower the loss).

    Optimizer Step (optimizer.step()): Train mode only. The optimizer actually turns the knobs to update the model.

    Zero Gradients (optimizer.zero_grad()): Train mode only. We wipe the slate clean for the next batch so the gradients don't stack up indefinitely.

By using torch.set_grad_enabled(phase == "train"), we tell PyTorch to completely ignore gradient tracking during the test phase, which saves massive amounts of memory and speeds up evaluation.

In [ ]:
from tqdm.notebook import trange, tqdm

def train(model, data_loader, criterion, optimizer, num_epochs=5):
    """
    Trains and evaluates the model for a specified number of epochs.
    """
    for epoch in trange(num_epochs, desc="Epochs"):
        print(f"\n--- Epoch {epoch + 1}/{num_epochs} ---")
        
        # Each epoch has a training and validation (test) phase
        for phase in ['train', 'test']:
            if phase == "train":
                model.train()  # Set model to training mode (enables Dropout, etc.)
            else:
                model.eval()   # Set model to evaluate mode (disables Dropout)
        
            # Variables to track our progress over the epoch
            running_loss = 0.0
            running_corrects = 0.0  
        
            # Iterate over the specific DataLoader (Train or Test)
            for data, target in tqdm(data_loader[phase], leave=False, desc=f"{phase.capitalize()} Phase"):
                # Move data and labels to the active device (CPU or GPU)
                data, target = data.to(device), target.to(device)

                # Only track gradients if we are in the training phase
                with torch.set_grad_enabled(phase == "train"): 
                    # 1. Forward Pass
                    output = model(data)
                    
                    # 2. Calculate Loss
                    loss = criterion(output, target)
                    
                    # 3. Get the prediction (the class with the highest probability)
                    preds = torch.argmax(output, 1)

                if phase == "train":
                    # 4. Backward pass: compute gradients
                    loss.backward()
                    
                    # 5. Update the model's weights
                    optimizer.step()
                    
                    # 6. Zero out the gradients for the next batch
                    optimizer.zero_grad()

                # Accumulate statistics for the current batch
                running_loss += loss.item() * data.size(0)
                running_corrects += torch.sum(preds == target.data).item()
            
            # Calculate the average loss and accuracy for the entire epoch
            epoch_loss = running_loss / len(data_loader[phase].dataset)
            epoch_acc = running_corrects / len(data_loader[phase].dataset)
            
            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}')

# NOTE: To actually start training, you will run this function:
# train(model, data_loader, criterion, optimizer, num_epochs=5)

Chapter 12: Save and Load the Model

Training a deep learning model can take hours, days, or even weeks depending on the dataset. If your computer crashes or loses power, you do not want to start from scratch.

To prevent this, we use Checkpoints.

In PyTorch, there are two ways to save a model:

    Saving the entire model object: This is generally discouraged because it can break if your directory structure changes, and it carries security vulnerabilities (arbitrary code execution) if you share it.

    Saving the state_dict: This is the professional standard. The state_dict is simply a Python dictionary that maps each layer of the model to its learned parameters (the weights and biases).

We will create two helper functions to easily save and load these dictionaries. We will also save an "Initial_Model" right now so we have a baseline of exactly what the model looked like before it learned anything.

In [ ]:
import os
import torch

# Create a local directory to store our saved models if it doesn't exist yet
checkpoint_dir = './model_checkpoint'
os.makedirs(checkpoint_dir, exist_ok=True)

def create_checkpoint(filename):
    """Saves the model's learned weights to a file."""
    filepath = os.path.join(checkpoint_dir, filename)
    # model.state_dict() extracts the current weights and biases
    torch.save(model.state_dict(), filepath)
    print(f"Model saved to {filepath}")

def load_checkpoint(filename):
    """Loads saved weights back into the model architecture."""
    filepath = os.path.join(checkpoint_dir, filename)
    # torch.load reads the file, load_state_dict applies it to the model
    model.load_state_dict(torch.load(filepath, map_location=device))
    print(f"Model loaded from {filepath}")

# Save the untrained model as a baseline
create_checkpoint("Initial_Model.pt")

Chapter 13: Train the Model for the First Time

It is finally time to hit the "Go" button! We will call our train() function to start the learning process.

Once the model is trained, looking at the overall accuracy isn't always enough. For example, if we achieve 90% overall accuracy, but the model gets 100% of the Pigeons right and 0% of the Eagles right, that is a heavily biased model!

To catch this, we will write a perClassAccuracy function. This will iterate through our testing dataset and calculate exactly how accurate the model is for each specific bird species.

In [ ]:
import numpy as np

# 1. Start the Training Process!
# We will run it for 1 epoch just to make sure everything works locally. 
# (You can increase num_epochs to 5 or 10 later to get a highly accurate model).
print("Starting training...")
train(model, data_loader, criterion, optimizer, num_epochs=1)

# 2. Save the model now that it has learned from 1 epoch
create_checkpoint("Initial_Model_v1.pt")

# NOTE: If you downloaded 'pre_trained_v1.pt' from the Educative sandbox and 
# placed it in your './model_checkpoint' folder, uncomment the line below to load it!
# load_checkpoint("pre_trained_v1.pt")

# 3. Define a function to check accuracy for each specific bird
def perClassAccuracy(model, classes):
    # Create arrays to keep track of correct guesses and total images per class
    class_correct = np.zeros(len(classes), dtype=np.int64)
    class_total = np.zeros_like(class_correct, dtype=np.int64)
    
    # Put model in evaluation mode
    model.eval()

    # Iterate through the unseen test data
    for data, target in data_loader["test"]:
        data, target = data.to(device), target.to(device)
        
        # Don't waste memory on gradients during testing
        with torch.set_grad_enabled(False):
            output = model(data)
            preds = torch.argmax(output, 1)
            
            # Compare predictions to the true labels
            for prediction, label in zip(preds, target.data):
                if prediction == label: 
                    class_correct[prediction] += 1
                class_total[label] += 1
                
    # Calculate the percentage
    per = np.round((100 * class_correct / class_total), 4)
    
    # Format the output cleanly
    out = "\n".join([f"{name} : {acc} %" for name, acc in zip(classes, per)])
    total_acc = 100 * sum(class_correct) / sum(class_total)
    
    return out + f"\n\nTotal Accuracy: {total_acc:.2f}%"

# 4. Print the final report
print("\n--- Per-Class Accuracy Report ---")
class_names = list(bird_dataset.classes_to_index.keys())
print(perClassAccuracy(model, class_names))

Chapter 14: Unfreeze 40% of the Model

Right now, only our custom classification head is learning. The entire ResNet-18 backbone is still frozen with its original ImageNet knowledge.

To achieve maximum accuracy, we want to perform Fine-Tuning. This means we will unfreeze the last few layers of the ResNet backbone. We don't unfreeze the early layers because they already know how to detect basic shapes and edges perfectly. But by unfreezing the later layers, we allow the model to adjust its high-level feature detection specifically for bird shapes (like beaks, wings, and talons).

In [ ]:
import numpy as np

def unfreeze(model, percent=0.25):
    """Unfreezes a specified percentage of the model's layers from the end backwards."""
    # Calculate how many layers to unfreeze
    num_layers_to_unfreeze = int(np.ceil(len(model._modules.keys()) * percent))
    
    # Get the names of the last 'n' layers
    layers_to_unfreeze = list(model._modules.keys())[-num_layers_to_unfreeze:] 
    print(f"Unfreezing these layers: {layers_to_unfreeze}")
    
    # Iterate through those specific layers and turn gradients back on
    for name in layers_to_unfreeze:
        for params in model._modules[name].parameters():
            params.requires_grad_(True)

def check_freeze(model):
    """Prints a status report of which layers are frozen and which are learning."""
    print("\n--- Model Layer Status ---")
    for name, layer in model._modules.items():
        # Check if the parameters in this layer require gradients
        param_status = [p.requires_grad for p in layer.parameters()]
        
        # If the layer has parameters, check if they are all True
        if param_status:
            is_learning = all(param_status)
            status = "Unfrozen (Learning)" if is_learning else "Frozen (Locked)"
            print(f"{name:15} -> {status}")

# Unfreeze the top 40% of the model
unfreeze(model, 0.40)

# Verify that the correct layers are now unfrozen
check_freeze(model)

Chapter 15 & 16: Learning Rate Schedulers

Until now, we used a fixed Learning Rate (LR). Think of the learning rate as the size of the steps the model takes down a mountain to find the lowest point (the lowest error).

    If the steps are too big, the model overshoots the minimum and fails to converge.

    If the steps are too small, it takes forever to train and might get stuck in a "local minimum" (a small valley that isn't the true bottom).

To solve this, we use a Scheduler to change the learning rate dynamically during training.

    Linear Scheduling: The step size changes in a straight, predictable line.

    Cosine Scheduling: The step size follows a smooth curve, tapering off gently at the end, which is excellent for fine-tuning.

The formula for the Cosine Scheduler is:
lr=max_lr+0.5⋅(min_lr−max_lr)⋅(1+cos(π⋅total_stepscurrent_step​))

Let's write and visualize these schedulers!

In [ ]:
import matplotlib.pyplot as plt

def linear_scheduling(min_lr, max_lr, curr_iter, total_steps):
    """Linearly adjusts the learning rate."""
    pct = curr_iter / total_steps
    return (max_lr - min_lr) * pct + min_lr

def cosine_scheduling(min_lr, max_lr, curr_iter, total_steps):
    """Smoothly adjusts the learning rate using a cosine curve."""
    pct = curr_iter / total_steps
    cos_out = np.cos(np.pi * pct) + 1
    return max_lr + (min_lr - max_lr) / 2.0 * cos_out

# Let's visualize how the learning rate will change over 10 iterations
iterations = range(11)
linear_lrs = [linear_scheduling(1, 5, i, 10) for i in iterations]
cosine_lrs = [cosine_scheduling(1, 5, i, 10) for i in iterations]

plt.figure(figsize=(10, 4))

# Plot Linear
plt.subplot(1, 2, 1)
plt.title('Linear Scheduling')
plt.xlabel('Iterations')
plt.ylabel('Learning Rate')
plt.plot(iterations, linear_lrs, "b-+")

# Plot Cosine
plt.subplot(1, 2, 2)
plt.title('Cosine Scheduling')
plt.xlabel('Iterations')
plt.plot(iterations, cosine_lrs, "r-+")

plt.tight_layout()
plt.show()

Chapter 17: Write the LR Finder Algorithm

Now we know how to schedule the learning rate, but how do we know what values to use for min_lr and max_lr?

We use an LR Finder Algorithm. This clever technique starts the learning rate at a microscopic value (like 1e-7) and rapidly increases it with every batch. We log the loss at each step.

    At first, the loss won't change (LR is too small).

    Then, the loss will start dropping rapidly (The "sweet spot").

    Finally, the LR becomes too huge, the model panics, and the loss explodes upward.

The ideal min_lr and max_lr are located right in that "sweet spot" where the loss was falling fastest.

In [ ]:
from pathlib import Path

class LRFinder:
    def __init__(self, model, min_lr=1e-7):
        self.model = model
        self.min_lr = min_lr
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.Adam(model.parameters(), lr=self.min_lr)
        
    def lrfind(self, trainLoader, max_lr=10, num_iter=200, loss_smoothing_beta=0.98, early_stop_loss_threshold=10):
        # 1. Save the model's current state so we don't ruin it during this wild test
        self.save_file = Path("tmpfile.pt")
        torch.save(self.model.state_dict(), self.save_file)
        
        self.history = {"lr": [], "losses": []}
        best_loss = 0
        avg_loss = 0
        iterator = iter(trainLoader)
        
        self.model.train() # Ensure we are in training mode
        
        for curr_iter in trange(num_iter, desc="Finding LR"):
            try:
                # Get the next batch of data
                data, target = next(iterator)
                data, target = data.to(device), target.to(device)
                            
                # Calculate loss and update weights
                loss = self._train_batch(data, target)

                # Generate the new learning rate using our cosine function
                update_lr = cosine_scheduling(self.min_lr, max_lr, curr_iter, num_iter)
                
                # Apply the new learning rate to the optimizer
                for g in self.optimizer.param_groups: 
                    g['lr'] = update_lr
                    
                self.history["lr"].append(update_lr)
                
                # Smooth the loss curve to make the graph readable
                avg_loss = loss_smoothing_beta * avg_loss + (1 - loss_smoothing_beta) * loss
                smoothed_loss = avg_loss / (1 - loss_smoothing_beta ** (curr_iter + 1))

                # Track the best loss
                if smoothed_loss < best_loss or curr_iter == 1:
                    best_loss = smoothed_loss

                self.history["losses"].append(smoothed_loss)
                
                # Emergency Stop: If the loss explodes, stop the test early
                if curr_iter > 1 and smoothed_loss > early_stop_loss_threshold * best_loss:
                    print("Stopping early: the loss has diverged (LR became too large).")
                    break

            except StopIteration:
                pass # End of dataloader
            
        # 2. Reset the model back to its original, safe state
        self.model.load_state_dict(torch.load(self.save_file, map_location=device))
        self.save_file.unlink() # Delete the temporary file
        
        # 3. Plot the results
        self._plot()

    def _train_batch(self, data, target):
        output = self.model(data)
        loss = self.criterion(output, target)
        loss.backward()
        self.optimizer.step()
        self.optimizer.zero_grad()
        return loss.item()

    def _plot(self):
        losses = self.history["losses"]
        lr = self.history["lr"]
        plt.figure(figsize=(10, 5))
        plt.semilogx(lr, losses) # Plot with a logarithmic X-axis
        plt.xlabel("Learning Rate (Log Scale)")
        plt.ylabel("Loss")
        plt.title("Learning Rate Finder")
        plt.show()

# Run the LR Finder
lr_finder = LRFinder(model)
lr_finder.lrfind(data_loader["train"])

Chapter 18: Model Training with Cosine Scheduler

Now we are going to update our original train function to support the scheduler.

Notice that because we are fine-tuning the backbone layers, we need to use a very small learning rate. The backbone already has valuable information in it; if we hit it with a massive learning rate, we will cause "catastrophic forgetting"—the model will literally forget what a bird looks like!

If the LR Finder graph gave you wonky numbers, a standard safe bet for fine-tuning ResNet is a base rate around 1e-4.

In [ ]:
from functools import partial

# --- UPDATED TRAINING FUNCTION ---
def train_with_scheduler(model, data_loader, criterion, optimizer, scheduler, num_epochs=5):
    current_iter = 0 # Track the total number of batches processed
    
    for epoch in trange(num_epochs, desc="Epochs"):
        print(f"\n--- Epoch {epoch + 1}/{num_epochs} ---")
        
        for phase in ['train', 'test']:
            if phase == "train":
                model.train()
            else:
                model.eval()
        
            running_loss = 0.0
            running_corrects = 0.0  
            
            for data, target in tqdm(data_loader[phase], leave=False, desc=f"{phase.capitalize()} Phase"):
                data, target = data.to(device), target.to(device)

                with torch.set_grad_enabled(phase == "train"): 
                    output = model(data)
                    loss = criterion(output, target)
                    preds = torch.argmax(output, 1)

                if phase == "train":
                    loss.backward()
                    optimizer.step()
                    optimizer.zero_grad()
                    
                    # UPDATE THE LEARNING RATE EVERY BATCH!
                    for g in optimizer.param_groups: 
                        g['lr'] = scheduler(current_iter)
                    current_iter += 1
                        
                running_loss += loss.item() * data.size(0)
                running_corrects += torch.sum(preds == target.data).item()
            
            epoch_loss = running_loss / len(data_loader[phase].dataset)
            epoch_acc = running_corrects / len(data_loader[phase].dataset)
            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}')


# --- EXECUTE THE FINE-TUNING ---

# We will use a safe base LR for fine-tuning
lr_finder_lr = 1e-4
min_lr = lr_finder_lr / 2
max_lr = lr_finder_lr * 2
num_epochs = 1 # Set to 3 or 5 for best results!

# Create our scheduler
# We use partial to lock in the min, max, and total steps, leaving only current_iter to be updated in the loop
total_training_steps = len(data_loader["train"]) * num_epochs
partial_cosine_scheduling = partial(cosine_scheduling, min_lr, max_lr, total_steps=total_training_steps)

# Re-initialize the optimizer with weight decay (L2 Regularization) to prevent overfitting
optimizer = optim.Adam(model.parameters(), lr=min_lr, weight_decay=1e-5)

print("Starting Fine-Tuning with Cosine Scheduler...")
train_with_scheduler(model, data_loader, criterion, optimizer, partial_cosine_scheduling, num_epochs=num_epochs)

# Print Final Accuracy and Save!
print("\n--- Final Per-Class Accuracy Report ---")
class_names = list(bird_dataset.classes_to_index.keys())
print(perClassAccuracy(model, class_names))

create_checkpoint("Final_Model_v2.pt")

Chapter 19: The OneCycle Policy

We have already tried a static learning rate and a custom cosine scheduler. Now, we will use PyTorch's built-in OneCycleLR scheduler.

The One Cycle Policy (which leads to a phenomenon called "Superconvergence") was introduced by researcher Leslie Smith. Instead of just decreasing the learning rate over time, the One Cycle policy does something counter-intuitive:

    Phase 1 (Warm-up): It starts at a low learning rate and increases it to a very high maximum. This acts as a regularization technique, preventing the model from getting stuck in sharp, narrow valleys of loss.

    Phase 2 (Annealing): It then decreases the learning rate smoothly back down to a tiny value so the model can gently settle into the best possible minimum.

To see the true power of this, we are going to reset our model back to its untrained state, unfreeze 40% of it, and let OneCycle do its magic.

In [ ]:
from torch.optim.lr_scheduler import OneCycleLR

print("Resetting model to baseline...")
# Load the completely untrained model we saved in Chapter 12
# (Note: make sure the filename matches exactly what you saved!)
load_checkpoint("Initial_Model.pt")

# Unfreeze the top 40% of the ResNet backbone for fine-tuning
unfreeze(model, 0.40)

# Define our optimizer with a base learning rate and weight decay (regularization)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

# Define the OneCycleLR Scheduler
# pct_start=0.3 means it will spend the first 30% of training increasing the LR, and 70% decreasing it
scheduler = OneCycleLR(
    optimizer,
    max_lr=0.001,
    epochs=5,
    steps_per_epoch=len(data_loader["train"]),
    pct_start=0.3,
    anneal_strategy='cos',
    div_factor=25.0, 
    final_div_factor=10000.0, 
    three_phase=True
)

print("OneCycle Policy Scheduler initialized!")

Chapter 20: Train the Model Using OneCycle Policy

Because we are using a built-in PyTorch scheduler instead of our custom mathematical function, our training loop gets slightly simpler.

PyTorch schedulers have a .step() method. All we have to do is call scheduler.step() immediately after we update our weights with optimizer.step(). PyTorch will handle all the complex math behind the scenes to adjust the learning rate for the next batch.

Let's train the model using this new policy and see how it performs!

In [ ]:
# --- ONE CYCLE TRAINING FUNCTION ---
def train_one_cycle(model, data_loader, criterion, optimizer, scheduler, num_epochs=5):
    for epoch in trange(num_epochs, desc="Epochs"):
        print(f"\n--- Epoch {epoch + 1}/{num_epochs} ---")
        
        for phase in ['train', 'test']:
            if phase == "train":
                model.train()
            else:
                model.eval()
        
            running_loss = 0.0
            running_corrects = 0.0  
            
            for data, target in tqdm(data_loader[phase], leave=False, desc=f"{phase.capitalize()} Phase"):
                data, target = data.to(device), target.to(device)

                with torch.set_grad_enabled(phase == "train"): 
                    output = model(data)
                    loss = criterion(output, target)
                    preds = torch.argmax(output, 1)

                if phase == "train":
                    loss.backward()
                    optimizer.step()
                    optimizer.zero_grad()
                    
                    # PYTORCH BUILT-IN SCHEDULER STEP
                    scheduler.step()
                        
                running_loss += loss.item() * data.size(0)
                running_corrects += torch.sum(preds == target.data).item()
            
            epoch_loss = running_loss / len(data_loader[phase].dataset)
            epoch_acc = running_corrects / len(data_loader[phase].dataset)
            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}')


# Execute the training! 
# We run it for 2 epochs here to see quick results, but you can increase it to 5!
print("Starting Superconvergence Training with OneCycleLR...")
train_one_cycle(model, data_loader, criterion, optimizer, scheduler, num_epochs=2)

# Print Final Accuracy and Save
print("\n--- Final OneCycle Per-Class Accuracy Report ---")
class_names = list(bird_dataset.classes_to_index.keys())
print(perClassAccuracy(model, class_names))

create_checkpoint("Final_Model_OneCycle.pt")

Chapter 21: Comparative Study and Future Prospects

Congratulations on reaching the end of the project! Let's review the journey of your model's architecture:
📈 The Training Journey

    Baseline Transfer Learning: We started by freezing the entire ResNet-18 backbone and just training a custom fully-connected head. This gave us a solid baseline of around 80% accuracy.

    Fine-Tuning with Custom Scheduling: By using the LR finder, a custom cosine scheduler, and unfreezing 40% of the backbone, we allowed the model to adapt its deeper feature extraction to birds, pushing accuracy to ~87%.

    Superconvergence: Finally, by implementing the OneCycle policy on the unfrozen backbone, we achieved remarkable improvements, reaching ~90% accuracy with fewer training epochs.

🚀 Future Work for Computer Vision

To push this model even further, you could:

    Clean the Data: Perform thorough data cleaning and verify label correctness.

    Upscale the Architecture: Swap ResNet-18 for larger models like ResNet-34, ResNet-50, or DenseNet.

    Data Augmentation: Add transformations like random flips, rotations, or color jittering in the BirdDataset class to make the model more robust.

🧠 Transfer Learning Beyond Images (NLP)

The transfer learning concepts you mastered here apply directly to Natural Language Processing (NLP) with models like BERT or GPT:

    Feature Extraction: Using a pre-trained language model's hidden layers to generate text embeddings.

    Fine-Tuning: Adapting a massive pre-trained language model on a specific, smaller dataset (like training a general chatbot to become a medical assistant).

    Multitask Learning: Training a model to perform translation, summarization, and sentiment analysis simultaneously.

    Knowledge Distillation: Taking a massive, heavy model (the "Teacher") and using it to train a much smaller, faster model (the "Student") so it can run on mobile devices.

Course Complete! 🎉